In [3]:
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
from selenium.webdriver.common.keys import Keys
import pandas as pd 
import random
import time

/Users/jongho/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [4]:
# 복권 사이트에서 당첨결과 엑셀 파일 다운로드

# Service 객체 생성
service = webdriver.chrome.service.Service(ChromeDriverManager().install())
# Service 객체를 webdriver.Chrome() 메서드에 전달
driver = webdriver.Chrome(service=service)
# Chrome 실행 파일 경로 설정
webdriver.chrome.service.DEFAULT_EXECUTABLE_PATH = '/Applications/Google Chrome.app/Contents/MacOS/Google Chrome'

url = 'https://dhlottery.co.kr/gameResult.do?method=win720'
driver.get(url)

# 다운받을 회차 범위
select = Select(driver.find_element(By.CSS_SELECTOR, "#drwNoStart"))
select.select_by_visible_text("1")
select = Select(driver.find_element(By.CSS_SELECTOR, "#drwNoEnd"))
# select.select_by_visible_text(str(end))
driver.find_element(By.CSS_SELECTOR, "#exelBtn").send_keys(Keys.ENTER)

print("파일 다운로드 완료!")
time.sleep(3)
driver.quit()

파일 다운로드 완료!


In [8]:
excel_file_path = "/Users/jongho/Downloads/pension.xlsb"
df = pd.read_excel(excel_file_path, header=2, dtype={'Unnamed: 1': float, '(100 X 10⩼/td>': str})
df

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,(100 X 10⩼/td>
0,2024.0,207.0,20240418.0,3v,211988.0,11988.0,1988.0,988.0,88.0,8.0,396540
1,NaN,NaN,NaN,211988,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,206.0,20240411.0,3v,489075.0,89075.0,9075.0,75.0,75.0,5.0,587695
3,NaN,NaN,NaN,489075,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,205.0,20240404.0,1v,472377.0,72377.0,2377.0,377.0,77.0,7.0,935471
...,...,...,...,...,...,...,...,...,...,...,...
409,NaN,NaN,NaN,544955,NaN,NaN,NaN,NaN,NaN,NaN,NaN
410,NaN,2.0,20200514.0,2v,450558.0,50558.0,558.0,558.0,58.0,8.0,154457
411,NaN,NaN,NaN,450558,NaN,NaN,NaN,NaN,NaN,NaN,NaN
412,NaN,1.0,20200507.0,4v,162132.0,62132.0,2132.0,132.0,32.0,2.0,278239


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 414 entries, 0 to 413
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Unnamed: 0      5 non-null      float64
 1   Unnamed: 1      207 non-null    float64
 2   Unnamed: 2      207 non-null    float64
 3   Unnamed: 3      414 non-null    object 
 4   Unnamed: 4      207 non-null    float64
 5   Unnamed: 5      207 non-null    float64
 6   Unnamed: 6      207 non-null    float64
 7   Unnamed: 7      207 non-null    float64
 8   Unnamed: 8      207 non-null    float64
 9   Unnamed: 9      207 non-null    float64
 10  (100 X 10⩼/td>  207 non-null    float64
dtypes: float64(10), object(1)
memory usage: 35.7+ KB


In [9]:
# 엑셀 파일을 열어서 데이터프레임으로 바꾸기
excel_file_path = "/Users/jongho/Downloads/pension.xlsb"
df = pd.read_excel(excel_file_path, header=2, dtype={'Unnamed: 1': float, '(100 X 10⩼/td>': str})

new_df = df[['Unnamed: 1', '(100 X 10⩼/td>']]
new_df = new_df.dropna()
new_df.columns = ['round', 'numbers']

for i in range(0, 6):
    new_df["n"+str(i+1)] = new_df['numbers'].apply(lambda x: x[i])

new_df = new_df.reset_index()

# print(new_df)

# 숫자별 당첨 확률 계산
def counts(i):
    value_counts = new_df[i].value_counts().sort_index().reset_index().rename(columns={i: 'number', 'count': i})

    return value_counts

for i in ['n1', 'n2', 'n3', 'n4', 'n5', 'n6']:
    if i == 'n1':
        number_prob_df = counts(i)
    else:
        number_prob_df = pd.merge(number_prob_df, counts(i), on='number')

total_count = max(new_df['round'])

for i in range(1, 7):
    number_prob_df["p"+str(i)] = number_prob_df["n"+str(i)] / total_count

# print(number_prob_df)

# 숫자별 당첨 확률을 가중치로 반영해서 무작위 숫자 추출
def rand(i):
    random_numbers = [random.random() for _ in range(len(number_prob_df))]
    number_prob_df['random'] = random_numbers
    number_prob_df["rand_"+str(i)] = number_prob_df["p"+str(i)] * number_prob_df['random']
    result = number_prob_df.loc[number_prob_df["rand_"+str(i)] == number_prob_df["rand_"+str(i)].max(), 'number']
    return result.tolist()

selected_numbers = []

for i in range(1, 7):
    top = rand(i)
    selected_numbers.append(top)

selected_numbers = sum(selected_numbers, [])

# 결과 출력
print(f"분석회차 : {min(new_df.index+1)}회 ~ {max(new_df.index+1)}회")
print(f"추출번호: {selected_numbers}")

분석회차 : 1회 ~ 207회
추출번호: ['2', '3', '6', '6', '5', '0']
